# Explaination

Why use ANN-based weightage instead of manually selecting combinations?

Manual combination testing gives separate scores, but it does not learn a general rule for how detector, recognizer, and scale influence performance together. ANN-based weighting uses known ground-truth identities in scene images to learn this relationship directly from data.

Key benefits of learned weightage:

- **Data-driven decisions:** The system learns from measured precision/recall/F1 outcomes, not intuition.
- **Captures interactions:** Detector, recognizer, and scale interact; ANN can model these dependencies better than manual weighting.
- **Reduced bias and better consistency:** Learned weights are reproducible for the same training data, unlike subjective manual tuning.
- **Interpretable importance:** Weightages indicate which detector and recognizer choices contribute most to identification performance.
- **Scales to larger search spaces:** As the number of models and settings grows, manual comparison becomes difficult, while the ANN approach remains structured and maintainable.

Why set weightages?

Weightages quantify how strongly each pipeline component contributes to final recognition quality. This helps justify model choices scientifically, prioritize optimization effort, and provide a clear, defensible methodology in reports.

# Model Combinations (ANN Weighting)

This notebook evaluates DeepFace detector + recognizer + scale combinations and then trains a **simple ANN** to learn which combinations perform best.

It uses known scene-image identities (from filename tokens) as supervision and outputs learned weightage for detector, recognizer, and scale.

In [ ]:
# Install runtime packages in the active notebook kernel (run once).
# This cell chooses a compatible TensorFlow stack based on Python version.
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version


def is_installed(dist_name: str) -> bool:
    try:
        version(dist_name)
        return True
    except PackageNotFoundError:
        return False


def pip_install(*packages: str) -> None:
    cmd = [sys.executable, "-m", "pip", "install", "-U", *packages]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)


def pip_uninstall_if_present(*packages: str) -> None:
    installed = [pkg for pkg in packages if is_installed(pkg)]
    if not installed:
        print("No existing ML packages to uninstall.")
        return
    cmd = [sys.executable, "-m", "pip", "uninstall", "-y", *installed]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)


py = sys.version_info
is_arm_mac = sys.platform == "darwin" and platform.machine() == "arm64"

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

pip_install("pip", "setuptools<82", "wheel")
pip_uninstall_if_present(
    "tensorflow",
    "tensorflow-macos",
    "tensorflow-metal",
    "keras",
    "tf-keras",
    "deepface",
    "retinaface",
    "retina-face",
)

if is_arm_mac and py < (3, 13):
    print("Installing Apple Silicon TensorFlow 2.16 stack for Python < 3.13")
    pip_install("protobuf>=3.20.3,<5")
    pip_install("tensorflow-macos==2.16.2", "tensorflow-metal==1.2.0", "tf-keras==2.16.0")
    pip_install("torch", "deepface==0.0.99", "retina-face==0.0.17")
else:
    print("Installing standard TensorFlow stack for Python 3.13+")
    pip_install("protobuf>=6.31.1,<7")
    pip_install("tensorflow==2.20.0", "tf-keras==2.20.1", "keras>=3.10.0")
    pip_install("torch", "deepface==0.0.99", "retina-face==0.0.17")

print("Restart the kernel after this cell finishes.")

In [ ]:
# -----------------------------
# Imports
# -----------------------------
# itertools: Cartesian products for all detector/recognizer/scale combinations
# re: parse identity letters from scene filenames
# Path: platform-safe file and folder handling
import gc
import importlib.abc
import itertools
import re
import sys
from pathlib import Path

# On this Python 3.13 Anaconda kernel, TensorFlow/Keras can segfault if
# pandas discovers the optional pyarrow package during import. We block
# pyarrow for this notebook process so TensorFlow can load safely.
if sys.version_info >= (3, 13):
    class _BlockPyArrow(importlib.abc.MetaPathFinder):
        def find_spec(self, fullname, path=None, target=None):
            if fullname == "pyarrow" or fullname.startswith("pyarrow."):
                raise ModuleNotFoundError(
                    "pyarrow disabled for this notebook to avoid a TensorFlow import crash"
                )
            return None

    sys.meta_path.insert(0, _BlockPyArrow())
    for name in list(sys.modules):
        if name == "pyarrow" or name.startswith("pyarrow."):
            del sys.modules[name]
    print("pyarrow disabled in-process for TensorFlow compatibility on Python 3.13")

# OpenCV for image read/resize
import cv2
# NumPy for vectors and math
import numpy as np
# Pandas for tabular tracking of results
import pandas as pd
# PyTorch for library-based ANN training
import torch
import torch.nn as nn
# DeepFace for detection + embedding extraction
from deepface import DeepFace
# Progress bars for long loops
from tqdm.auto import tqdm

# -----------------------------
# Global configuration
# -----------------------------
# Notebook assumes it is run from project root.
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "open_data_set"

# Candidate detector backends to compare.
DETECTORS = ["opencv", "mtcnn", "retinaface", "ssd", "mediapipe"]
# Candidate recognizer models to compare.
RECOGNIZERS = ["ArcFace", "Facenet512", "SFace", "GhostFaceNet", "Dlib"]
# Manual upscaling factors (applied before detection).
SCALES = [1.0, 1.5, 2.0, 3.0]

# A predicted identity is accepted only if nearest gallery distance <= threshold.
MATCH_THRESHOLD = 0.35

# Use up to 20 gallery images per identity for training/evaluation.
MAX_GALLERY_PER_ID = 20

# Stability controls for heavy DeepFace loops.
# Start safe, then increase once stable.
MAX_SCENES = 120          # set None for full dataset run
MAX_FACES_PER_SCENE = 12  # cap detections processed per scene
MAX_SCENE_SIDE = 1400     # downscale huge images before detection
CLEAR_TF_EVERY_COMBO = True

# Fail early if dataset is missing.
assert DATASET_ROOT.exists(), f"Dataset folder not found: {DATASET_ROOT}"

# Show available GPU backends.
print(f"PyTorch MPS available: {torch.backends.mps.is_available()}")
try:
    import tensorflow as tf
    print("TensorFlow devices:", tf.config.list_physical_devices())
except Exception as e:
    print("TensorFlow device query skipped:", e)

In [ ]:
# -----------------------------
# Helper functions
# -----------------------------
def list_images(folder: Path):
    """Return all supported image files inside a folder."""
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])


def infer_single_identity(path: Path) -> str:
    """
    For gallery files, identity is taken from filename prefix.
    Example: 'c_12.jpg' -> 'C'.
    """
    token = path.stem.split("_")[0]
    return token[:1].upper()


def infer_scene_identities(path: Path):
    """
    For scene files, filename token may contain multiple people.
    Example: 'cd_gp_0_eo_12.jpg' -> {'C', 'D'}.
    """
    token = path.stem.split("_")[0].upper()
    ids = set(re.findall(r"[A-Z]", token))
    return ids


def cosine_distance(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    """Cosine distance (smaller means more similar)."""
    a_norm = np.linalg.norm(a) + eps
    b_norm = np.linalg.norm(b) + eps
    return float(1.0 - np.dot(a, b) / (a_norm * b_norm))


def precision_recall_f1(expected_ids, predicted_ids):
    """
    Set-based metrics on identities per scene:
    - expected_ids: true identities present in scene
    - predicted_ids: identities predicted by pipeline
    """
    tp = len(expected_ids & predicted_ids)
    fp = len(predicted_ids - expected_ids)
    fn = len(expected_ids - predicted_ids)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


def clear_runtime_memory():
    """Best-effort cleanup to reduce memory pressure during long runs."""
    gc.collect()
    try:
        import tensorflow as tf
        tf.keras.backend.clear_session()
    except Exception:
        pass

# -----------------------------
# Build gallery + scene tables
# -----------------------------
# Match FAISS notebook behavior exactly:
# - gallery from photos_all_faces
# - scenes from photos_all
# so both notebooks evaluate on the same image pool.
gallery_dir = DATASET_ROOT / "photos_all_faces"
scene_dir = DATASET_ROOT / "photos_all"

if not scene_dir.exists():
    raise FileNotFoundError(f"Scene folder not found: {scene_dir}")

print(f"Using scene folder (FAISS-aligned): {scene_dir}")

gallery_paths = list_images(gallery_dir)
scene_paths = list_images(scene_dir)

# Optional quick-run mode: keep only first N scenes.
if MAX_SCENES is not None:
    scene_paths = scene_paths[:MAX_SCENES]

# Gallery dataframe: one row per known identity image.
gallery_rows = []
for p in gallery_paths:
    gallery_rows.append({"path": str(p), "identity": infer_single_identity(p)})
gallery_df = pd.DataFrame(gallery_rows)

# Enforce max images per identity to control runtime and class balance.
gallery_df = gallery_df.groupby("identity", group_keys=False).head(MAX_GALLERY_PER_ID).reset_index(drop=True)

# Scene dataframe: each row includes expected IDs (ground truth).
scene_rows = []
for p in scene_paths:
    expected = infer_scene_identities(p)
    if expected:
        scene_rows.append({"path": str(p), "expected_ids": expected})
scene_df = pd.DataFrame(scene_rows)

print(f"Gallery images used: {len(gallery_df)}")
print(f"Scene images used: {len(scene_df)}")
print("Expected scene IDs example:", scene_df.iloc[0]["expected_ids"] if len(scene_df) else set())

In [ ]:
# -----------------------------
# Build gallery embedding cache
# -----------------------------
# Why cache?
# Every combination reuses the same gallery, but each recognizer has different
# embedding space. So we compute gallery embeddings once per recognizer model,
# then reuse during combination scoring.
def l2_normalize(vec: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Return an L2-normalized float32 vector for fast cosine comparisons."""
    arr = np.asarray(vec, dtype=np.float32)
    norm = max(float(np.linalg.norm(arr)), eps)
    return arr / norm


gallery_cache = {}

for model_name in RECOGNIZERS:
    rows = []
    print(f"Building gallery cache for model: {model_name}")

    # Iterate through known gallery faces and extract embeddings.
    for row in tqdm(gallery_df.itertuples(index=False), total=len(gallery_df), desc=f"gallery:{model_name}"):
        try:
            # detector_backend='skip' because gallery images are already face crops.
            reps = DeepFace.represent(
                img_path=row.path,
                model_name=model_name,
                detector_backend="skip",
                enforce_detection=False,
            )
            if len(reps) == 0:
                continue

            emb = l2_normalize(reps[0]["embedding"])
            rows.append({
                "identity": row.identity,
                "embedding": emb,
                "path": row.path,
            })
        except Exception:
            # Keep loop robust if one image/model call fails.
            continue

    gallery_model_df = pd.DataFrame(rows)
    if len(gallery_model_df) > 0:
        gallery_matrix = np.stack(gallery_model_df["embedding"].to_list()).astype(np.float32)
        gallery_identities = gallery_model_df["identity"].to_numpy()
    else:
        gallery_matrix = np.empty((0, 0), dtype=np.float32)
        gallery_identities = np.empty((0,), dtype=object)

    gallery_cache[model_name] = {
        "df": gallery_model_df,
        "matrix": gallery_matrix,
        "identities": gallery_identities,
    }
    print(f"  cached embeddings: {len(gallery_model_df)}")

In [ ]:
# -----------------------------
# Combination scoring pipeline
# -----------------------------
def detect_faces_scaled(scene_bgr, detector_backend: str, scale: float):
    """
    Detect faces after optional upscaling.
    Includes size guards to reduce memory pressure.
    """
    # Resize very large scenes down before any upscaling to avoid OOM.
    h0, w0 = scene_bgr.shape[:2]
    largest_side = max(h0, w0)
    if largest_side > MAX_SCENE_SIDE:
        shrink = MAX_SCENE_SIDE / float(largest_side)
        scene_bgr = cv2.resize(scene_bgr, dsize=None, fx=shrink, fy=shrink, interpolation=cv2.INTER_AREA)

    # Optional upscaling for detector sensitivity.
    if scale != 1.0:
        resized = cv2.resize(scene_bgr, dsize=None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    else:
        resized = scene_bgr

    faces = DeepFace.extract_faces(
        img_path=resized,
        detector_backend=detector_backend,
        enforce_detection=False,
        align=True,
    )
    return faces


def nearest_gallery_identity(face_embedding: np.ndarray, gallery_entry: dict):
    """Fast nearest-neighbor lookup using a prebuilt normalized gallery matrix."""
    gallery_matrix = gallery_entry["matrix"]
    if gallery_matrix.size == 0:
        return None, float("inf")

    emb = l2_normalize(face_embedding)
    similarities = gallery_matrix @ emb
    best_idx = int(np.argmax(similarities))
    best_dist = float(1.0 - similarities[best_idx])
    if best_dist <= MATCH_THRESHOLD:
        return gallery_entry["identities"][best_idx], best_dist
    return None, best_dist


# Each row in combo_rows will represent one (detector, recognizer, scale) tuple.
combo_rows = []
combo_list = list(itertools.product(DETECTORS, RECOGNIZERS, SCALES))
detector_scale_list = list(itertools.product(DETECTORS, SCALES))
recognizers_with_gallery = [
    recognizer for recognizer in RECOGNIZERS
    if gallery_cache.get(recognizer, {}).get("matrix", np.empty((0, 0), dtype=np.float32)).size > 0
]
combo_metrics = {
    (detector, recognizer, scale): []
    for detector, recognizer, scale in combo_list
    if recognizer in recognizers_with_gallery
}

print(f"Scoring {len(combo_list)} combinations on {len(scene_df)} scenes")
print(f"Stability settings -> MAX_SCENES={MAX_SCENES}, MAX_FACES_PER_SCENE={MAX_FACES_PER_SCENE}, MAX_SCENE_SIDE={MAX_SCENE_SIDE}")
print(
    f"Face-detection passes reduced from {len(combo_list) * len(scene_df)} "
    f"to {len(detector_scale_list) * len(scene_df)} by reusing detections across recognizers."
)

# Sweep detector/scale first so detected face crops can be reused by all recognizers.
for detector, scale in tqdm(detector_scale_list, desc="Detector-scale sweeps"):
    for scene_row in tqdm(
        scene_df.itertuples(index=False),
        total=len(scene_df),
        leave=False,
        desc=f"{detector}@{scale}",
    ):
        expected_ids = set(scene_row.expected_ids)
        scene_bgr = cv2.imread(scene_row.path)
        if scene_bgr is None:
            continue

        try:
            faces = detect_faces_scaled(scene_bgr, detector, scale)
        except Exception:
            faces = []

        face_crops = []
        for face_item in faces[:MAX_FACES_PER_SCENE]:
            face_img = face_item.get("face", None)
            if face_img is None or np.size(face_img) == 0:
                continue
            face_crops.append(face_img)

        # Drop scene arrays aggressively to lower RAM spikes.
        del scene_bgr

        for recognizer in recognizers_with_gallery:
            gallery_entry = gallery_cache[recognizer]
            predicted_ids = set()

            for face_img in face_crops:
                try:
                    # detector_backend='skip' because face_img is already a cropped face.
                    rep = DeepFace.represent(
                        img_path=face_img,
                        model_name=recognizer,
                        detector_backend="skip",
                        enforce_detection=False,
                    )
                    if len(rep) == 0:
                        continue

                    best_identity, _ = nearest_gallery_identity(
                        np.array(rep[0]["embedding"], dtype=np.float32),
                        gallery_entry,
                    )
                    if best_identity is not None:
                        predicted_ids.add(best_identity)
                except Exception:
                    continue

            precision, recall, f1 = precision_recall_f1(expected_ids, predicted_ids)
            combo_metrics[(detector, recognizer, scale)].append({
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "num_predicted_ids": len(predicted_ids),
                "num_expected_ids": len(expected_ids),
            })

    # Cleanup now happens once per detector/scale sweep so model reuse stays fast.
    if CLEAR_TF_EVERY_COMBO:
        clear_runtime_memory()

for detector, recognizer, scale in combo_list:
    scene_metrics = combo_metrics.get((detector, recognizer, scale), [])
    if len(scene_metrics) == 0:
        continue

    mdf = pd.DataFrame(scene_metrics)
    combo_rows.append({
        "detector": detector,
        "recognizer": recognizer,
        "scale": scale,
        "mean_precision": mdf["precision"].mean(),
        "mean_recall": mdf["recall"].mean(),
        "mean_f1": mdf["f1"].mean(),
        "mean_num_predicted_ids": mdf["num_predicted_ids"].mean(),
    })

# Final score table sorted by ground-truth mean F1.
combo_df = pd.DataFrame(combo_rows).sort_values("mean_f1", ascending=False).reset_index(drop=True)
print(f"Valid combinations scored: {len(combo_df)}")
combo_df.head(10)

In [ ]:
# -----------------------------
# Simple ANN (PyTorch implementation)
# -----------------------------
# Goal: learn a smooth mapping
#   (detector, recognizer, scale) -> expected quality (mean_f1)
# This uses a library ANN (PyTorch), as requested.

# Create index maps for one-hot encoding categorical features.
detector_to_idx = {d: i for i, d in enumerate(DETECTORS)}
recognizer_to_idx = {r: i for i, r in enumerate(RECOGNIZERS)}

num_detector = len(DETECTORS)
num_recognizer = len(RECOGNIZERS)
# +1 for the numeric scale feature.
input_dim = num_detector + num_recognizer + 1

# X = encoded combination features, y = measured combo mean_f1.
X = np.zeros((len(combo_df), input_dim), dtype=np.float32)
y = combo_df["mean_f1"].values.astype(np.float32).reshape(-1, 1)

# Encode each combination row.
for i, row in combo_df.iterrows():
    # Detector one-hot
    X[i, detector_to_idx[row["detector"]]] = 1.0
    # Recognizer one-hot (offset by detector block length)
    X[i, num_detector + recognizer_to_idx[row["recognizer"]]] = 1.0
    # Scale normalized to [0, 1]
    X[i, -1] = row["scale"] / max(SCALES)

# Random split for lightweight train/test reporting.
rng = np.random.default_rng(42)
idx = np.arange(len(X))
rng.shuffle(idx)
split = max(1, int(0.8 * len(X)))
train_idx, test_idx = idx[:split], idx[split:]

X_train_np, y_train_np = X[train_idx], y[train_idx]
if len(test_idx) > 0:
    X_test_np, y_test_np = X[test_idx], y[test_idx]
else:
    X_test_np = np.empty((0, input_dim), dtype=np.float32)
    y_test_np = np.empty((0, 1), dtype=np.float32)

# Select device (Apple GPU via MPS if available, else CPU).
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"ANN device: {device}")

# Convert NumPy arrays to torch tensors on selected device.
X_train = torch.from_numpy(X_train_np).to(device)
y_train = torch.from_numpy(y_train_np).to(device)
X_test = torch.from_numpy(X_test_np).to(device)
y_test = torch.from_numpy(y_test_np).to(device)


class SimpleANN(nn.Module):
    """Small 1-hidden-layer regressor for combination score prediction."""
    def __init__(self, in_dim: int, hidden_dim: int = 12):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


# Build model, loss, and optimizer.
torch.manual_seed(42)
model = SimpleANN(input_dim, hidden_dim=12).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

epochs = 1500

# Training loop.
model.train()
for ep in range(epochs):
    optimizer.zero_grad()
    y_hat = model(X_train)
    loss = criterion(y_hat, y_train)
    loss.backward()
    optimizer.step()

    if ep % 300 == 0 or ep == epochs - 1:
        print(f"epoch={ep:4d} train_mse={loss.item():.6f}")


@torch.no_grad()
def predict_ann(X_in_np):
    """Inference helper returning NumPy predictions for consistency."""
    model.eval()
    x_t = torch.from_numpy(X_in_np.astype(np.float32)).to(device)
    pred = model(x_t).detach().cpu().numpy()
    return pred


# Report basic fit quality.
train_pred = predict_ann(X_train_np)
train_mse = float(np.mean((train_pred - y_train_np) ** 2))
print(f"\nTrain MSE: {train_mse:.6f}")

if len(X_test_np) > 0:
    test_pred = predict_ann(X_test_np)
    test_mse = float(np.mean((test_pred - y_test_np) ** 2))
    print(f"Test MSE:  {test_mse:.6f}")
else:
    print("Test set empty (very small number of valid combinations).")

In [ ]:
# -----------------------------
# Learned weightage extraction
# -----------------------------
# Estimate feature importance from the first PyTorch layer.
# Intuition: larger absolute input weights imply stronger influence
# on hidden activations and therefore on output score.
with torch.no_grad():
    fc1_w = model.fc1.weight.detach().cpu().numpy()  # shape: [hidden_dim, input_dim]
feature_importance = np.mean(np.abs(fc1_w), axis=0)  # per-input-feature importance

# Split importances back into feature groups.
detector_importance = feature_importance[:num_detector]
recognizer_importance = feature_importance[num_detector:num_detector + num_recognizer]
scale_importance = feature_importance[-1]


def to_percentage(arr):
    """Normalize raw importances into percentages for readability."""
    arr = np.array(arr, dtype=np.float64)
    s = arr.sum()
    if s <= 0:
        return np.zeros_like(arr)
    return 100.0 * arr / s


# Weightage percentages within each categorical group.
detector_weight_pct = to_percentage(detector_importance)
recognizer_weight_pct = to_percentage(recognizer_importance)

# Build pretty tables for report output.
detector_weight_df = pd.DataFrame({
    "detector": DETECTORS,
    "weight_percent": detector_weight_pct,
}).sort_values("weight_percent", ascending=False)

recognizer_weight_df = pd.DataFrame({
    "recognizer": RECOGNIZERS,
    "weight_percent": recognizer_weight_pct,
}).sort_values("weight_percent", ascending=False)

print("Detector weightage (learned):")
display(detector_weight_df)

print("Recognizer weightage (learned):")
display(recognizer_weight_df)

# Scale is a single numeric feature, so we show its raw magnitude.
print(f"Scale feature importance (raw): {scale_importance:.6f}")

# ANN-based ranking: this is the model's predicted quality per combo.
combo_df = combo_df.copy()
combo_df["ann_predicted_f1"] = predict_ann(X).reshape(-1)
combo_df = combo_df.sort_values("ann_predicted_f1", ascending=False).reset_index(drop=True)

print("Top combinations by ANN-predicted F1:")
combo_df.head(15)